# RETECO · Replication on MTRAG: precision of the qrels label

Repeats the paper's main experiment on a second dataset, **MTRAG** (Katsis et al., TACL 2025:
110 conversations, 842 turns, 4 domains, human answerability for every turn, reference answers).
Claim: the label "no gold passage in the top 5 ⇒ context is insufficient" loses precision as
retrieval gets stronger (retrieved non-gold passages often contain the answer), and this changes
the conclusions about reader abstention. The results are Section 7 (Replication on MTRAG)
and Table 2 of the paper.

1. **Data**: files from `github.com/IBM/mt-rag-benchmark` at a pinned commit (`MTRAG_REV`):
   passage_level corpora clapnq, cloud, fiqa, govt (366k passages), tasks, qrels, annotator
   judgments. The label analysis uses the 777 ANSWERABLE + PARTIAL turns (they have qrels).
2. **Retrievers**, top 100: BM25 over all user questions, BM25 over the last turn, BM25 over
   MTRAG's rewritten question, dense `Qwen3-Embedding-0.6B` over the rewrite, and the hybrid of
   BM25 and dense over the rewrite (min-max, 0.5/0.5, fixed in advance: there is no train split).
   Metrics: nDCG@10, R@5. Table 2 of the paper orders the retrievers by nDCG@10.
3. **Contexts**: the top 5 of each retriever, plus "all gold" of the turn (the analogue of 2b).
4. **Judge** `microsoft/phi-4`: the same prompt and parsing as in `suff.ipynb`, with the MTRAG answer as
   reference: yes / partial / no for each context.
5. **Human check of the judge**: each passage on its own, both gold (qrels) and the passages that
   MTRAG annotators explicitly marked as not relevant. Gives P(judge yes | annotator: not relevant).
6. **Readers** OCC-RAG-1.7B and Qwen3-4B-Instruct-2507 answer from each context, with the same classes,
   prompts and p_answerable as in `generate.ipynb`; plus 55 unanswerable turns on the hybrid top 5
   (they have no MTRAG rewrites, so the hybrid query there = all user questions).
7. **Summary** with bootstrap (1000, seed 0): share of qrels *none*, judge labels, false misses,
   P/R of the *none* label, human check, reader refusals and AUROC of p_answerable under both labels.

**How to run.** Runtime → **A100** (phi-4 in bf16 ≈29 GB) → "Run all". If cell 2
asks for a restart, restart the session and "Run all" again. If the run is interrupted, "Run all"
again: runs, vectors, labels and answers already on Drive are skipped. To skip the readers, set
`READERS = []`: the label summary is computed before the readers and saved separately from them.

**Time (A100, estimate).** MTRAG files ≈130 MB: 1–2 min. BM25 with three query types: 5–10 min.
Vectors for 366k passages (up to 512 tokens, ≈250 on average) with the 0.6B model: 25–45 min
(once; ≈750 MB on Drive).
phi-4 judge: 777 × 6 contexts ≈ 4.7k prompts of 2–5k tokens + ≈2.6k short ones
(single passage): 45–75 min. Readers: 2 × ≈4.7k answers: OCC 15–25 min, Qwen3-4B 25–45 min.
**Total ≈2.5–3.5 h**, without readers ≈1.5–2 h. The actual speed is printed as the run progresses.

**Output.** All in `reteco/mtrag/` on Drive:
- `runs/<retriever>/<domain>/run.trec` + `config.json`; `retrieval.csv`: nDCG@10, R@5, R@100 (macro and micro)
- `suff/phi-4_<retriever|gold>.jsonl`: judge label, rationale, evidence_docs, raw response, qrels label;
  `suff/phi-4_single.jsonl`: single passages with the human label
- `gen/<reader>_<retriever|gold|unans_hybrid>.jsonl`: status, p_answerable, answer, qrels label
- `summary.csv`: all numbers with 95% CIs; `turns.jsonl`, `passages.jsonl`: turns and the texts of all
  passages in the contexts (for local analysis without the corpus); `unanswerable_hybrid_top5.jsonl`

**Next.** Copy `reteco/mtrag/` (without `index/`) from Drive to `data/drive/reteco/mtrag/` of the repo.

**To keep the session from disconnecting when idle**: after "Run all", open the browser console
(Cmd+Option+J → Console), paste the code below and press Enter. Once a minute it clicks the
resources button while the runtime is connected, and it stops by itself once the notebook has finished
and released the GPU (it does not reconnect, so no compute units are spent). The computer must not
go to sleep, and the tab must stay open.
A log of all output with timestamps is written to `logs/` in the project folder on Drive (`DRIVE`).

```js
// Clicker for Colab: keeps the session from disconnecting when idle while the runtime is connected.
// Paste it into the console of the Colab tab (Cmd+Option+J → Console tab → paste → Enter)
// after the notebook has been started. Once a minute it clicks the resources button in the header.
// It stops by itself once the runtime has disconnected (the notebook finished or crashed and
// released the GPU) and does NOT reconnect, so no compute units are spent on an idle machine.
(() => {
  if (window.__retecoKeepAlive) { console.log("clicker is already running"); return; }
  const btn = () => document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
  const disconnected = () => {
    const b = btn();
    if (!b) return true;
    // the button also shows the GPU type ("Reconnect T4"), so only the first word is compared;
    // the non-English labels below are the same button texts in the Russian-locale Colab UI
    const text = (b.textContent || "").trim().split(/\s+/)[0].toLowerCase();
    const tip = (b.getAttribute("tooltiptext") || "").toLowerCase();
    return ["connect", "reconnect", "подключиться", "переподключиться"].includes(text)
      || tip.startsWith("connect to a new") || tip.startsWith("подключиться к");
  };
  let seen = false, clicks = 0;
  window.__retecoKeepAlive = setInterval(() => {
    if (!disconnected()) {
      seen = true;
      btn().click();                       // opens/closes the resources panel; harmless
      clicks += 1;
      console.log(`keep-alive ${new Date().toLocaleTimeString()} · ${clicks}`);
    } else if (seen) {
      clearInterval(window.__retecoKeepAlive);
      window.__retecoKeepAlive = null;
      console.log("runtime disconnected: clicker stopped");
    }
  }, 60 * 1000);
  console.log("clicker started: once a minute; stops by itself after the runtime disconnects");
})();
```

In [ ]:
# ── 1. Settings ──────────────────────────────────────────────────────────────
MTRAG_REV = "53fef6f5917c7e10c436f697e69a955dbc795d81"   # IBM/mt-rag-benchmark commit (main as of 2026-10-10)
MT_DOMAINS = ["clapnq", "cloud", "fiqa", "govt"]         # in the MTRAG tasks cloud is called ibmcloud
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMB_BATCH = 64        # "CUDA out of memory" → 32
MAX_DOC_LEN = 512     # tokens per passage for the embedder (MTRAG passages are 512-token chunks)
MAX_Q_LEN = 512
BM25_K1, BM25_B = 0.9, 0.4   # as in the RETECO kit's BM25; bm25s (Lucene), English stopwords, PyStemmer stemming
# Hybrid: min-max fusion of BM25 and dense on the rewritten question. The weight is fixed a priori, NOT tuned:
# MTRAG has no train split, and tuning on the same turns we evaluate on is not allowed.
HYB_W = 0.5           # BM25 weight; dense gets 1 − HYB_W
TOPK = 100            # run length
K_CTX = 5             # documents in the context, as in generate.ipynb and suff.ipynb
JUDGE = "microsoft/phi-4"   # sufficiency judge: same prompt and parsing as in suff.ipynb
READERS = ["occ-ai/OCC-RAG-1.7B", "Qwen/Qwen3-4B-Instruct-2507"]   # [] = labels only, no readers
DO_GOLD = True        # also the "all gold of the turn" context (analogue of 2b), for the judge and readers
DO_SINGLE = True      # human check of the judge: each passage on its own (gold and passages rejected by annotators)
SINGLE_CLASSES = ["ANSWERABLE"]   # human answerability classes of the turns included in the check
MAXLEN = 16384        # phi-4 allows no more; gold can have up to 9 passages
JUDGE_NEW = 400       # as MAX_NEW in suff.ipynb
READ_NEW = 1200       # as MAX_NEW in generate.ipynb: OCC writes its reasoning before the answer
TRIM_WORDS = 300      # if a prompt does not fit, each document is cut to this many words (as in suff/generate)
CHUNK = 256           # prompts between writes to Drive
N_BOOT, SEED = 1000, 0   # summary bootstrap: over turns within a domain, 95% percentile
DRIVE = "/content/drive/MyDrive/RETECO 2026/reteco"   # project folder on Drive; all MTRAG output goes to DRIVE/mtrag

In [ ]:
# ── 1b. Compute-unit safeguard ──────────────────────────────────────────────
# At the end of the notebook the runtime disconnects itself, so an idle GPU does not burn compute units.
# If a cell fails and nothing is run for {IDLE_STOP_MIN} min, it disconnects as well.
# To turn this off, set AUTO_STOP = False in the settings.
NB = "mtrag"

import threading, time
from IPython import get_ipython
AUTO_STOP = globals().get("AUTO_STOP", True)
IDLE_STOP_MIN = globals().get("IDLE_STOP_MIN", 15)

def stop_runtime(reason):
    print(f"\n[{time.strftime('%H:%M:%S')}] {reason}: disconnecting the runtime to save compute units", flush=True)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("could not disconnect:", e)

_WD = {"t": None}
def _pre_cell(*a, **k):
    if _WD["t"]:
        _WD["t"].cancel(); _WD["t"] = None
def _post_cell(result):
    if AUTO_STOP and getattr(result, "error_in_exec", None) is not None:
        _WD["t"] = threading.Timer(IDLE_STOP_MIN * 60, stop_runtime,
                                   args=(f"error and {IDLE_STOP_MIN} min without activity",))
        _WD["t"].daemon = True; _WD["t"].start()
_ip = get_ipython()
if _ip is not None and not getattr(_ip, "_reteco_guard", False):
    _ip.events.register("pre_run_cell", _pre_cell)
    _ip.events.register("post_run_cell", _post_cell)
    _ip._reteco_guard = True
print("safeguard:", "the runtime disconnects itself at the end or", IDLE_STOP_MIN, "min after an error"
      if AUTO_STOP else "off (AUTO_STOP = False)")

In [ ]:
# ── 2. Drive, vLLM, GPU ─────────────────────────────────────────────────────
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
# JAX (pulled in by bm25s) preallocates 75% of GPU memory by default, and vLLM then fails to start
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
from google.colab import drive
drive.mount("/content/drive")
!pip -q install vllm openai pytrec-eval-terrier 2>&1 | tail -2
!pip -q install bm25s sentence-transformers 2>&1 | tail -2
!pip -q install PyStemmer 2>&1 | tail -2
import torch

# vllm installs its own torch, while Colab's torchvision/torchaudio stay built for a different
# CUDA, and import vllm fails. Fix: uninstall torchaudio and reinstall torchvision
# for the new torch's CUDA. A runtime restart is needed afterwards.
def _imports_ok(mod):
    try:
        __import__(mod); return True
    except RuntimeError:
        return False       # installed, but built for a different CUDA
    except ImportError:
        return True        # not installed, so no conflict
fixed = False
if not _imports_ok("torchaudio"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchaudio"], check=True)
    fixed = True
if not _imports_ok("torchvision") and "+" in torch.__version__:
    base, cu = torch.__version__.split("+")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--force-reinstall", "--no-deps",
                    f"torchvision==0.{int(base.split('.')[1]) + 15}.*",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"], check=True)
    fixed = True
if fixed:
    raise RuntimeError("Dependencies fixed. Runtime → Restart session, "
                       "then Run all again.")
if globals().get("NEED_GPU", True):
    assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100 or L4"
MEM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if MEM and torch.cuda.get_device_capability(0)[0] >= 8 else "half"   # T4 (sm75): no bf16
BIG = MEM >= 39                                                             # A100 40/80 GB
MID = MEM >= 22                                                             # L4 24 GB and above
print(torch.cuda.get_device_name(0) if MEM else "no GPU", f"{MEM:.0f} GB, {DTYPE}, torch", torch.__version__)

def load_llm(model, maxlen, util=0.88):
    """Uses the real stdout/stderr while the constructor runs: in a Jupyter kernel vLLM fails
    on sys.stdout.fileno() before the weights are even loaded."""
    from vllm import LLM
    out, err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        return LLM(model=model, dtype=DTYPE, max_model_len=maxlen, gpu_memory_utilization=util, seed=0)
    finally:
        sys.stdout, sys.stderr = out, err

def chat(tok, msgs):
    """Prompt built with the model's chat template; Qwen3 thinking is disabled (fast, deterministic)."""
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
%%writefile /content/gen_worker.py
"""Answers of one reader for all jobs (2b, and 2c for each first stage). Run from
generate.ipynb as a separate process per reader, so that vLLM reliably frees GPU
memory before the next model. The summary cell also imports this module."""
import json, math, os, re, sys, time

OCC_STOP = [151643, 151645, 151683]   # <|endoftext|> <|im_end|> <|answer_end|>, from the OCC generation_config
CONTROL_SYSTEM = (
    "You are a helpful assistant in a multi-turn conversation. Answer the user's latest message "
    "using only the numbered documents attached to it.\n"
    "Reply in this format:\n"
    "Line 1: ANSWERABLE if the documents contain enough information to answer, otherwise UNANSWERABLE.\n"
    "Next lines: if ANSWERABLE, the answer in 2-5 sentences in a natural conversational tone; "
    "if UNANSWERABLE, say briefly what information is missing.")


def official_history(h, prev):
    """History limited to exactly what the official conversation_history contains: the
    (question, answer) pairs whose answer appears there; if older turns are compressed into
    "Earlier discussion covered: ...", that summary is prepended to the first question."""
    if not h or h.strip() == "No previous conversation.":
        return []
    hist = [(q, a) for q, a in prev if a and a in h]
    if "Earlier discussion covered" in h and hist:
        hist[0] = (h.split("\n\nRecent conversation:")[0].strip() + "\n\n" + hist[0][0], hist[0][1])
    return hist


def load_turns(data, domains, split):
    """Turns that have gold documents. history is the official dialogue history (official_history).
    reference is the gold answer of the CURRENT turn: used only by the audit and the judge,
    never in the reader prompt. Gold is only what appears in qrels (3 gold ids in politics
    are missing from the corpus). gold_in_prior counts the gold documents of this turn that
    were already gold for earlier turns: then the answer may be in the history."""
    out = []
    for dom in domains:
        qrels = {tuple(l.split()[0:3:2]) for l in open(f"{data}/{dom}/qrels_{split}.txt") if l.strip()}
        for c in json.load(open(f"{data}/{dom}/benchmark_{split}.json", encoding="utf-8")):
            prev, seen = [], set()
            for t in c["turns"]:
                qid = f"{c['id']}_turn_{t['turn_id']}"
                gold = list(dict.fromkeys(d for d in t.get("gold_doc_ids") or t.get("supporting_doc_ids") or []
                                          if (qid, d) in qrels))
                if gold:
                    out.append({"domain": dom, "topic_id": qid, "turn_id": t["turn_id"], "query": t["query"],
                                "history": official_history(t.get("conversation_history", ""), prev),
                                "gold": gold, "gold_in_prior": len(set(gold) & seen),
                                "reference": t.get("answer", "")})
                prev.append((t["query"], t.get("answer", "")))
                seen |= set(gold)
    return out


def read_top(path, k):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], []).append((int(p[3]), p[2]))
    return {q: [d for _, d in sorted(v)[:k]] for q, v in run.items()}


def contexts(C, turns, split, job):
    """"2b" → gold documents; "nodoc" → no documents (what the model knows on its own);
    "2c-<run>" → top-K from Drive/reteco/runs/<run>_<split>."""
    if job == "2b":
        return {(t["domain"], t["topic_id"]): t["gold"] for t in turns}
    if job == "nodoc":
        return {(t["domain"], t["topic_id"]): [] for t in turns}
    if job.startswith("ctrl-"):
        # judge-leniency control: top-K of the same stage for a turn from ANOTHER dialogue in the
        # same domain (half the dialogue list away, same turn number or else the first turn), with
        # this turn's own gold removed, so the context is known not to contain the answer
        tops = {dom: read_top(f"{C['drive']}/runs/{job[5:]}_{split}/{dom}/run.trec", 2 * C["k_ctx"])
                for dom in C["domains"]}
        out, by_dom = {}, {}
        for t in turns:
            by_dom.setdefault(t["domain"], []).append(t)
        for dom, ts in by_dom.items():
            dlg = sorted({t["topic_id"].rsplit("_turn_", 1)[0] for t in ts})
            ids = {t["topic_id"] for t in ts}
            for t in ts:
                d, k = t["topic_id"].rsplit("_turn_", 1)
                other = dlg[(dlg.index(d) + len(dlg) // 2) % len(dlg)]
                src = f"{other}_turn_{k}" if f"{other}_turn_{k}" in ids else f"{other}_turn_1"
                gold = set(t["gold"])
                out[(dom, t["topic_id"])] = [x for x in tops[dom].get(src, []) if x not in gold][:C["k_ctx"]]
        return out
    tops = {dom: read_top(f"{C['drive']}/runs/{job[3:]}_{split}/{dom}/run.trec", C["k_ctx"])
            for dom in C["domains"]}
    return {(t["domain"], t["topic_id"]): tops[t["domain"]].get(t["topic_id"], []) for t in turns}


def sufficiency(gold, ctx):
    """Number of gold documents in the context, and the label: all = all gold that fits in K; none = no gold."""
    n = len(set(gold) & set(ctx))
    full = min(len(gold), len(ctx)) if ctx else len(gold)
    return n, ("none" if n == 0 else "all" if n >= full else "part")


def corpus_texts(data, need):
    """need: {domain: {doc_id}} → {(domain, doc_id): text}."""
    out = {}
    for dom, ids in need.items():
        for ln in open(f"{data}/{dom}/documents.jsonl", encoding="utf-8"):
            if ln.strip():
                d = json.loads(ln)
                if d["doc_id"] in ids:
                    out[(dom, d["doc_id"])] = d["content"]
    return out


def history_msgs(t):
    m = []
    for q, a in t["history"]:
        m += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return m


class OCC:
    """The OCC chat template itself places the documents in the last user turn; generation
    starts with <|query_analysis_start|>, the status is <|status_start|>(UN)ANSWERABLE<|status_end|>."""
    stop, skip_special, marker = OCC_STOP, False, "<|status_start|>"
    STATUS = re.compile(r"<\|status_start\|>(.*?)(?:<\|status_end\|>|$)", re.S)
    ANSWER = re.compile(r"<\|answer_start\|>(.*?)(?:<\|answer_end\|>|<\|im_end\|>|$)", re.S)
    SPECIAL = re.compile(r"<\|[a-z_]+\|>")

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        msgs = history_msgs(t) + [{"role": "user", "content": t["query"]}]
        return self.tok.apply_chat_template(msgs, documents=[{"text": d} for d in docs],
                                            add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        m = self.STATUS.search(raw)
        status = self.SPECIAL.sub("", m.group(1)).strip().upper() if m else ""
        m = self.ANSWER.search(raw)
        return status, (self.SPECIAL.sub("", m.group(1)).strip() if m else "")

    def prefix(self, raw):
        """Output up to the verdict position: the marker plus the newline after it (OCC always
        writes "<|status_start|>\nANSWERABLE"), so that the next token is the verdict."""
        i = raw.find(self.marker)
        if i < 0:
            return None
        j = i + len(self.marker)
        while j < len(raw) and raw[j] in "\r\n":
            j += 1
        return raw[:j]


class Control:
    """Qwen3 readers (the control prompt): same input, status on the first line."""
    stop, skip_special = None, True

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        numbered = "\n\n".join(f"[{i}] {d}" for i, d in enumerate(docs, 1)) or "(none)"
        msgs = ([{"role": "system", "content": CONTROL_SYSTEM}] + history_msgs(t)
                + [{"role": "user", "content": f"Documents:\n{numbered}\n\nQuestion: {t['query']}"}])
        return self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        first, _, rest = raw.strip().partition("\n")
        m = re.search(r"\b(UN)?ANSWERABLE\b", first.upper())
        if not m:
            return "", raw.strip()
        return m.group(0), rest.strip()

    def prefix(self, raw):
        """Everything before the status word on the first line (the model may start with "Line 1:" or **)."""
        lead = len(raw) - len(raw.lstrip())
        m = re.search(r"\b(?:UN)?ANSWERABLE\b", raw[lead:].split("\n", 1)[0], re.I)
        return raw[:lead + m.start()] if m else None


def p_status(tok, lp):
    """Probability of ANSWERABLE at the status position: summed probabilities of tokens that are
    prefixes of "ANSWERABLE" versus "UNANSWERABLE" among the top 20 (ANS... vs UN...). mass is
    how much probability fell on these two options at all."""
    a = u = 0.0
    for tid, x in lp.items():
        s = tok.decode([tid]).strip()
        if s and "UNANSWERABLE".startswith(s):
            u += math.exp(x.logprob)
        elif s and "ANSWERABLE".startswith(s):
            a += math.exp(x.logprob)
    return (round(a / (a + u), 6) if a + u > 0 else None), round(a + u, 6)


def main(reader_id, cfg_path):
    from vllm import LLM, SamplingParams
    C = json.load(open(cfg_path))
    short = reader_id.split("/")[-1]
    todo_any = False
    for split in C["splits"]:
        keys = {(t["domain"], t["topic_id"]) for t in load_turns(C["data"], C["domains"], split)}
        for job in C["jobs"]:
            path = f"{C['drive']}/gen/{short}_{job}_{split}.jsonl"
            done = ({(r["domain"], r["topic_id"]) for r in map(json.loads, open(path, encoding="utf-8"))}
                    if os.path.exists(path) else set())
            todo_any |= bool(keys - done)
    if not todo_any:
        print(f"{short}: everything is already done, skipping model load", flush=True)
        return
    llm = LLM(model=reader_id, dtype=C["dtype"], max_model_len=C["maxlen"],
              gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    R = (OCC if "OCC" in reader_id else Control)(tok)
    # OCC: greedy only, no repetition_penalty, otherwise the structured format breaks
    sp = SamplingParams(temperature=0.0, max_tokens=C["max_new"], stop_token_ids=R.stop,
                        skip_special_tokens=R.skip_special, spaces_between_special_tokens=False)
    sp1 = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
    short = reader_id.split("/")[-1]
    limit = C["maxlen"] - C["max_new"]
    ntok = lambda p: len(tok(p, add_special_tokens=False).input_ids)
    os.makedirs(f"{C['drive']}/gen", exist_ok=True)
    for split in C["splits"]:
        turns = load_turns(C["data"], C["domains"], split)
        jobs = [(j, contexts(C, turns, split, j)) for j in C["jobs"]]
        need = {}
        for _, ctx in jobs:
            for (dom, _), ids in ctx.items():
                need.setdefault(dom, set()).update(ids)
        text = corpus_texts(C["data"], need)
        for job, ctx in jobs:
            path = f"{C['drive']}/gen/{short}_{job}_{split}.jsonl"
            done = set()
            if os.path.exists(path):
                done = {(r["domain"], r["topic_id"]) for r in map(json.loads, open(path, encoding="utf-8"))}
            prompts, keep, skipped = [], [], 0
            for t in turns:
                if (t["domain"], t["topic_id"]) in done:
                    continue
                ids = ctx[(t["domain"], t["topic_id"])]
                docs = [text[(t["domain"], d)] for d in ids]
                p, trimmed = R.prompt(t, docs), False
                if ntok(p) > limit:      # does not fit: trim every document rather than drop gold
                    p, trimmed = R.prompt(t, [" ".join(d.split()[:C["trim_words"]]) for d in docs]), True
                if ntok(p) > limit:
                    skipped += 1
                    continue
                prompts.append(p)
                keep.append((t, ids, trimmed))
            print(f"{short} · {job} · {split}: to do {len(prompts)}, done {len(done)}, "
                  f"too long {skipped}", flush=True)
            t0 = time.time()
            with open(path, "a", encoding="utf-8") as f:
                for s in range(0, len(prompts), C["chunk"]):
                    outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
                    raws = [o.outputs[0].text for o in outs]
                    # second pass: prompt + answer up to the status position, one token, top-20 logprobs
                    pref = [(i, R.prefix(r)) for i, r in enumerate(raws)]
                    pref = [(i, x) for i, x in pref if x is not None]
                    probs = {}
                    if pref:
                        o1 = llm.generate([prompts[s + i] + x for i, x in pref], sp1, use_tqdm=False)
                        for (i, _), o in zip(pref, o1):
                            probs[i] = p_status(tok, o.outputs[0].logprobs[0])
                    for i, (o, raw) in enumerate(zip(outs, raws)):
                        t, ids, trimmed = keep[s + i]
                        status, answer = R.parse(raw)
                        n_in, suff = sufficiency(t["gold"], ids)
                        p, mass = probs.get(i, (None, 0.0))
                        f.write(json.dumps({
                            "domain": t["domain"], "topic_id": t["topic_id"], "turn_id": t["turn_id"],
                            "split": split, "reader": reader_id, "mode": job.split("-")[0],
                            "first_stage": job[3:] if job.startswith("2c-") else None,
                            "context_ids": ids, "n_gold": len(t["gold"]), "gold_in_prior": t["gold_in_prior"],
                            "gold_in_ctx": n_in, "suff": suff, "status": status, "answer": answer,
                            "p_answerable": p, "p_mass": mass, "finish": o.outputs[0].finish_reason,
                            "trimmed": trimmed, "raw": raw}, ensure_ascii=False) + "\n")
                    f.flush()
                    k, el = s + len(outs), time.time() - t0
                    print(f"  {k}/{len(prompts)} · {el / 60:.1f} min · ≈{el / k * (len(prompts) - k) / 60:.0f} min left",
                          flush=True)


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

In [ ]:
%%writefile /content/suff_prompt.py
import json, re

LABELS = ["yes", "partial", "no"]
SYSTEM = ("You are a careful annotator for a conversational question answering benchmark. "
          "You decide whether a set of documents contains the information needed to answer a question. "
          "You judge only by what the documents say, never by your own knowledge.")
TASK = """Decide whether the DOCUMENTS contain the information needed for the core claim(s) of the reference answer to the latest question.

Labels:
- "yes": the documents support the core claim(s) of the reference answer for the latest question. Missing minor details, examples or exact wording is fine.
- "partial": the documents support at least one core point, but a central part of the answer is missing.
- "no": the documents do not contain the answer. Text on the same topic, or text that only restates the question, without the actual answer counts as "no".
Rules:
- Judge only by the documents, not by your own knowledge.
- The earlier conversation is not evidence: if the answer appears only there and not in the documents, it does not count.
- answered_in_history: true if the earlier conversation already states the answer to the latest question, otherwise false.
- evidence_docs: the numbers of the documents that support the answer; an empty list if none.

Return only a JSON object with keys in this order: {"rationale": "<at most 2 sentences>", "label": "yes" | "partial" | "no", "answered_in_history": true | false, "evidence_docs": [<document numbers>]}"""

def suff_msgs(t, docs):
    hist = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in t["history"]) or "(none)"
    numbered = "\n\n".join(f"[{i}] {d}" for i, d in enumerate(docs, 1)) or "(none)"
    user = (f"## Conversation so far\n{hist}\n\n## Latest question\n{t['query']}\n\n"
            f"## Reference answer (written by an expert; use it only to know what the correct answer says)\n"
            f"{t['reference']}\n\n## Documents\n{numbered}\n\n{TASK}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def parse_suff(raw, n_docs):
    """First {...} in the response → fields; anything that fails to parse is None (the raw response is kept)."""
    m = re.search(r"\{.*\}", raw or "", re.S)
    try:
        d = json.loads(m.group(0))
        assert isinstance(d, dict)
    except Exception:
        return {"label": None, "answered_in_history": None, "evidence_docs": [], "rationale": None}
    lab = str(d.get("label", "")).strip().strip('"').lower()
    ah = d.get("answered_in_history")
    ah = {"true": True, "false": False}.get(ah.strip().lower()) if isinstance(ah, str) else ah
    ev = d.get("evidence_docs")
    nums = [str(x).strip("[] ") for x in (ev if isinstance(ev, list) else [])]   # 2, "2", "[2]"
    return {"label": lab if lab in LABELS else None,
            "answered_in_history": ah if isinstance(ah, bool) else None,
            "evidence_docs": sorted({int(x) for x in nums if x.isdigit() and 1 <= int(x) <= n_docs}),
            "rationale": str(d.get("rationale", ""))[:1000]}

In [ ]:
%%writefile /content/mtrag_worker.py
"""MTRAG: GPU steps run as separate processes, so that each model frees GPU memory
before the next one (like gen_worker in generate.ipynb). The notebook also imports this module.
    python mtrag_worker.py embed <cfg>           passage and query vectors → Drive (float16 .npy)
    python mtrag_worker.py judge <cfg>           sufficiency judge, suff.ipynb prompt → Drive
    python mtrag_worker.py read <reader> <cfg>   reader responses, classes and status as in gen_worker → Drive
A job is a jsonl line {"job", "key", "t": {history, query[, reference]}, "docs": [texts], "meta"};
keys already in the output file are skipped; results are written to Drive every `chunk` prompts."""
import json, os, re, sys, time, zipfile
import gen_worker as G
import suff_prompt as S


def jsonl(path):
    """Rows of a jsonl file; a truncated record (session died mid-line) is skipped."""
    out = []
    if os.path.exists(path):
        for ln in open(path, encoding="utf-8"):
            if ln.strip():
                try:
                    out.append(json.loads(ln))
                except json.JSONDecodeError:
                    pass
    return out


def append(path):
    """Open for appending; if the previous write stopped mid-line, add a newline first."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and os.path.getsize(path):
        with open(path, "rb") as f:
            f.seek(-1, 2)
            tail = f.read(1)
        if tail != b"\n":
            with open(path, "a", encoding="utf-8") as f:
                f.write("\n")
    return open(path, "a", encoding="utf-8")


def norm(text):
    """Collapse runs of blank lines into one: cloud/govt have dozens of them, wasting prompt tokens."""
    return re.sub(r"\n\s*\n", "\n\n", (text or "").strip())


def load_corpus(repo, dom):
    """Domain passages read directly from corpora/passage_level/<domain>.jsonl.zip. Row:
    {"_id","id","url","text","title"}; the title is already the first line of text; _id = id in qrels."""
    z = zipfile.ZipFile(f"{repo}/corpora/passage_level/{dom}.jsonl.zip")
    name = max((n for n in z.namelist() if n.endswith(".jsonl")), key=lambda n: z.getinfo(n).file_size)
    ids, texts = [], []
    with z.open(name) as f:
        for ln in f:
            if ln.strip():
                d = json.loads(ln)
                ids.append(str(d.get("_id") or d["id"]))
                texts.append(norm(d.get("text")))
    return ids, texts


def todo_by_out(items, out_of):
    """[(output file, jobs whose key is not done yet, number already done)] in order of first appearance."""
    groups = {}
    for it in items:
        groups.setdefault(out_of(it), []).append(it)
    work = []
    for path, its in groups.items():
        done = {r.get("key") for r in jsonl(path)}
        work.append((path, [it for it in its if it["key"] not in done], len(done)))
    return work


def fit(tok, build, docs, limit, trim_words):
    """Build the prompt; if it does not fit, trim each document to trim_words words (as in generate/suff); if it still does not fit, None."""
    ntok = lambda p: len(tok(p, add_special_tokens=False).input_ids)
    p = build(docs)
    if ntok(p) <= limit:
        return p, False
    p = build([" ".join(d.split()[:trim_words]) for d in docs])
    return (p, True) if ntok(p) <= limit else (None, True)


def progress(name, k, n, t0):
    el = time.time() - t0
    print(f"  {name}: {k}/{n} · {el / 60:.1f} min · ≈{el / max(k, 1) * (n - k) / 60:.0f} min left", flush=True)


def embed(C):
    import numpy as np, torch, transformers
    from packaging.version import Version
    from sentence_transformers import SentenceTransformer
    idx, block = C["index"], 20000
    docs_todo = [d for d in C["domains"] if not os.path.exists(f"{idx}/{d}.npy")]
    q_todo = [d for d in C["domains"] if not os.path.exists(C["q_path"][d])]
    if not docs_todo and not q_todo:
        print("embedder: everything is already done, model not loaded", flush=True)
        return
    dk = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
    model = SentenceTransformer(C["model"], device="cuda", trust_remote_code=True,
                                model_kwargs={dk: torch.bfloat16 if C["dtype"] == "bfloat16" else torch.float16})
    qp = (model.prompts or {}).get("query") or ("Instruct: Given a web search query, retrieve relevant "
                                                "passages that answer the query\nQuery:")
    dp = (model.prompts or {}).get("document") or ""
    print("query prompt:", repr(qp), "· document prompt:", repr(dp), flush=True)

    def enc(texts, prompt, max_len):
        model.max_seq_length = max_len
        e = model.encode(texts, prompt=prompt, batch_size=C["batch"], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)
        assert not np.isnan(e).any(), "NaN in vectors"
        return e.astype(np.float16)

    def save(path, arr):
        with open(path + ".tmp", "wb") as f:
            np.save(f, arr)
        os.replace(path + ".tmp", path)

    for dom in docs_todo:
        ids, texts = load_corpus(C["repo"], dom)
        t0, parts = time.time(), []
        for b in range(0, len(texts), block):     # in blocks: an interruption loses at most one block
            pp = f"{idx}/{dom}.part{b // block:03d}.npy"
            if not os.path.exists(pp):
                save(pp, enc(texts[b:b + block], dp, C["max_doc_len"]))
            parts.append(pp)
            progress(f"vectors {dom}", min(b + block, len(texts)), len(texts), t0)
        json.dump(ids, open(f"{idx}/{dom}.ids.json", "w"))
        save(f"{idx}/{dom}.npy", np.concatenate([np.load(p) for p in parts]))
        for p in parts:
            os.remove(p)
        print(f"{dom}: {len(texts)} passages in {(time.time() - t0) / 60:.1f} min", flush=True)
    for dom in q_todo:
        qs = C["queries"][dom]
        save(C["q_path"][dom], enc([t for _, t in qs], qp, C["max_q_len"]) if qs else np.zeros((0, 1), np.float16))
    json.dump({"model": C["model"], "query_prompt": qp, "doc_prompt": dp, "max_doc_len": C["max_doc_len"],
               "max_q_len": C["max_q_len"], "dtype": C["dtype"], "stored": "float16, L2-normalized",
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
               "date": time.strftime("%Y-%m-%dT%H:%M:%S")},
              open(f"{idx}/config.json", "w"), ensure_ascii=False, indent=2)


def judge(C):
    from vllm import LLM, SamplingParams
    short = C["judge"].split("/")[-1]
    work = todo_by_out(jsonl(C["judge_items"]), lambda it: f"{C['out']}/suff/{short}_{it['job']}.jsonl")
    if not any(its for _, its, _ in work):
        print("judge: everything is already done, model not loaded", flush=True)
        return
    llm = LLM(model=C["judge"], dtype=C["dtype"], max_model_len=C["maxlen"], gpu_memory_utilization=0.9, seed=0)
    tok = llm.get_tokenizer()
    sp = SamplingParams(temperature=0.0, max_tokens=C["judge_new"])
    chat = lambda msgs: tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False,
                                                enable_thinking=False)
    for path, its, nd in work:
        prompts, keep, skipped = [], [], 0
        for it in its:
            p, trimmed = fit(tok, lambda docs, t=it["t"]: chat(S.suff_msgs(t, docs)), it["docs"],
                             C["maxlen"] - C["judge_new"], C["trim_words"])
            if p is None:
                skipped += 1
                continue
            prompts.append(p)
            keep.append((it, trimmed))
        name = os.path.basename(path)
        print(f"{name}: to do {len(prompts)}, done {nd}, did not fit {skipped}", flush=True)
        t0 = time.time()
        for s in range(0, len(prompts), C["chunk"]):
            outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
            with append(path) as f:
                for (it, trimmed), o in zip(keep[s:s + C["chunk"]], outs):
                    raw = o.outputs[0].text
                    f.write(json.dumps({"key": it["key"], **it["meta"], "judge": C["judge"],
                                        **S.parse_suff(raw, len(it["docs"])), "raw": raw, "trimmed": trimmed},
                                       ensure_ascii=False) + "\n")
            progress(name, s + len(outs), len(prompts), t0)


def read(reader_id, C):
    from vllm import LLM, SamplingParams
    short = reader_id.split("/")[-1]
    work = todo_by_out(jsonl(C["read_items"]), lambda it: f"{C['out']}/gen/{short}_{it['job']}.jsonl")
    if not any(its for _, its, _ in work):
        print(f"{short}: everything is already done, model not loaded", flush=True)
        return
    llm = LLM(model=reader_id, dtype=C["dtype"], max_model_len=C["maxlen"], gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    R = (G.OCC if "OCC" in reader_id else G.Control)(tok)
    # OCC: greedy only, no repetition_penalty; otherwise the structured format breaks (as in gen_worker)
    sp = SamplingParams(temperature=0.0, max_tokens=C["read_new"], stop_token_ids=R.stop,
                        skip_special_tokens=R.skip_special, spaces_between_special_tokens=False)
    sp1 = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
    for path, its, nd in work:
        prompts, keep, skipped = [], [], 0
        for it in its:
            p, trimmed = fit(tok, lambda docs, t=it["t"]: R.prompt(t, docs), it["docs"],
                             C["maxlen"] - C["read_new"], C["trim_words"])
            if p is None:
                skipped += 1
                continue
            prompts.append(p)
            keep.append((it, trimmed))
        name = os.path.basename(path)
        print(f"{name}: to do {len(prompts)}, done {nd}, did not fit {skipped}", flush=True)
        t0 = time.time()
        for s in range(0, len(prompts), C["chunk"]):
            outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
            raws = [o.outputs[0].text for o in outs]
            # second pass, as in gen_worker: prompt + response up to the status position, one token, top-20 logprobs
            pref = [(i, R.prefix(r)) for i, r in enumerate(raws)]
            pref = [(i, x) for i, x in pref if x is not None]
            probs = {}
            if pref:
                o1 = llm.generate([prompts[s + i] + x for i, x in pref], sp1, use_tqdm=False)
                for (i, _), o in zip(pref, o1):
                    probs[i] = G.p_status(tok, o.outputs[0].logprobs[0])
            with append(path) as f:
                for i, (o, raw) in enumerate(zip(outs, raws)):
                    it, trimmed = keep[s + i]
                    status, answer = R.parse(raw)
                    p, mass = probs.get(i, (None, 0.0))
                    f.write(json.dumps({"key": it["key"], **it["meta"], "reader": reader_id, "status": status,
                                        "answer": answer, "p_answerable": p, "p_mass": mass,
                                        "finish": o.outputs[0].finish_reason, "trimmed": trimmed, "raw": raw},
                                       ensure_ascii=False) + "\n")
            progress(name, s + len(outs), len(prompts), t0)


if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "read":
        read(sys.argv[2], json.load(open(sys.argv[3])))
    else:
        {"embed": embed, "judge": judge}[mode](json.load(open(sys.argv[2])))

In [ ]:
# ── 3. Log, TREC runs, GPU steps as separate processes ─────────────────────
import os, re, sys, json, time, csv, hashlib, subprocess, urllib.request
import numpy as np, pytrec_eval
sys.path.insert(0, "/content")
import gen_worker as G, suff_prompt as S, mtrag_worker as W
OUT = f"{DRIVE}/mtrag"
for _d in ("runs", "suff", "gen", "index"):
    os.makedirs(f"{OUT}/{_d}", exist_ok=True)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Writes to a temporary file first, then renames it, so an interrupted write never
    leaves a partial file. Scores are written strictly decreasing and with spare digits:
    pytrec_eval breaks score ties by docno in reverse order, which in five domains would
    systematically rank _neg_ above _doc_. Tied scores keep the dict order (that is,
    the first-stage order)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            last = None
            for r, (d, sc) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                sc = float(sc)
                if last is not None and sc >= last:
                    sc = last - 1e-9 * max(1.0, abs(last))
                last = sc
                f.write(f"{q}\tQ0\t{d}\t{r}\t{sc:.12g}\t{tag}\n")
    os.replace(path + ".tmp", path)


def run_path(name, dom):
    return f"{OUT}/runs/{name}/{dom}/run.trec"

def stream(cmd):
    """Stream the child process output into the cell line by line (as in generate.ipynb)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "VLLM_LOGGING_LEVEL": "WARNING"})
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def free_gpu(min_free_gb=34):
    """Before vLLM: kill leftover GPU processes (except the kernel) and wait for free memory.
    Without this the judge failed to start: only 9 of 40 GB were free on the A100 after the embedder."""
    import torch
    if not torch.cuda.is_available():
        return
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout
    except FileNotFoundError:
        out = ""
    for pid in {int(x) for x in out.split() if x.strip().isdigit()} - {os.getpid()}:
        try:
            os.kill(pid, 9); print("killed GPU process:", pid)
        except OSError:
            pass
    for _ in range(60):
        free, total = torch.cuda.mem_get_info()
        if free / 2**30 >= min(min_free_gb, 0.9 * total / 2**30):
            break
        time.sleep(2)
    print(f"GPU: {free / 2**30:.1f} of {total / 2**30:.1f} GB free")

def save_jsonl(path, rows):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(path + ".tmp", path)

# ── log: everything the cells print, timestamped → Drive/reteco/logs ────────
import sys
class _Tee:
    def __init__(self, out, path):
        self.out, self.f, self.bol = out, open(path, "a", encoding="utf-8"), True
    def write(self, x):
        self.out.write(x)
        try:
            for part in x.splitlines(True):
                if self.bol:
                    self.f.write(time.strftime("%m-%d %H:%M:%S "))
                self.f.write(part)
                self.bol = part.endswith("\n")
            self.f.flush()
        except Exception:
            pass
        return len(x)
    def flush(self):
        self.out.flush()
    def __getattr__(self, k):
        return getattr(self.out, k)
os.makedirs(f"{DRIVE}/logs", exist_ok=True)
if not isinstance(getattr(sys.stdout, "_reteco_log", None), str):
    LOG_PATH = f"{DRIVE}/logs/{globals().get('NB', 'notebook')}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    sys.stdout = _Tee(sys.stdout, LOG_PATH)
    sys.stdout._reteco_log = LOG_PATH
LOG_PATH = sys.stdout._reteco_log
print("log:", LOG_PATH)

In [ ]:
# ── 4. MTRAG data: repository files, turns, qrels, annotator judgments ──────
REPO = "/content/mt-rag-benchmark"
RAW = f"https://raw.githubusercontent.com/IBM/mt-rag-benchmark/{MTRAG_REV}"
H = "mtrag-human"
FILES = ([f"corpora/passage_level/{d}.jsonl.zip" for d in MT_DOMAINS]
         + [f"{H}/generation_tasks/reference.jsonl", f"{H}/conversations/conversations.json"]
         + [f"{H}/retrieval_tasks/{d}/{x}" for d in MT_DOMAINS
            for x in (f"{d}_lastturn.jsonl", f"{d}_questions.jsonl", f"{d}_rewrite.jsonl", "qrels/dev.tsv")])

def fetch(rel):
    path = f"{REPO}/{rel}"
    if os.path.exists(path):
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for attempt in range(5):
        try:
            urllib.request.urlretrieve(f"{RAW}/{rel}", path + ".tmp")
            os.replace(path + ".tmp", path)
            return path
        except Exception as e:
            print("  download failed:", rel, type(e).__name__, e)
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"failed to download {RAW}/{rel}")

t0 = time.time()
for rel in FILES:
    fetch(rel)
print(f"MTRAG files @{MTRAG_REV[:7]}: {len(FILES)} in {time.time() - t0:.0f} s")

def strip_user(s):
    """'|user|: a\n|user|: b' → 'a\nb': the speaker tag is noise for BM25 and the embedder."""
    return "\n".join(p.strip() for p in re.split(r"(?:^|\n)\|user\|:", s or "") if p.strip())

def dom_of(collection):
    """mt-rag-ibmcloud-elser-512-100-20240502 → cloud: the tasks say ibmcloud, the paths and corpus say cloud."""
    names = {"clapnq": "clapnq", "ibmcloud": "cloud", "cloud": "cloud", "fiqa": "fiqa", "govt": "govt"}
    hit = [names[p] for p in str(collection).lower().split("-") if p in names]
    assert hit, f"unknown collection {collection}"
    return hit[0]

def qa_pairs(msgs):
    """input up to the current question → [[question, agent answer]]: the history as the judge and readers see it."""
    out = []
    for m in msgs:
        if m["speaker"] == "user":
            out.append([m["text"].strip(), ""])
        elif out:
            out[-1][1] = (out[-1][1] + "\n" + m["text"].strip()).strip()
    return out

def page_of(dom, pid):
    """Source page of a passage: the id without chunk offsets; for clapnq also without the paragraph offset
    (822086267_7384-8758-0-1374 → 822086267, ibmcld_00474-7885-8455 → ibmcld_00474)."""
    p = pid.split("-")[0]
    return p.split("_")[0] if dom == "clapnq" else p

# qrels (BEIR tsv with a header): query-id = <conversation_id><::><turn>
QRELS = {d: {} for d in MT_DOMAINS}
for d in MT_DOMAINS:
    for ln in open(f"{REPO}/{H}/retrieval_tasks/{d}/qrels/dev.tsv", encoding="utf-8"):
        p = ln.rstrip("\n").split("\t")
        if len(p) >= 3 and p[0] != "query-id" and float(p[2]) > 0:
            QRELS[d].setdefault(p[0], {})[p[1]] = int(float(p[2]))
QFILE = {v: {} for v in ("lastturn", "questions", "rewrite")}
for d in MT_DOMAINS:
    for v in QFILE:
        for ln in open(f"{REPO}/{H}/retrieval_tasks/{d}/{d}_{v}.jsonl", encoding="utf-8"):
            if ln.strip():
                r = json.loads(ln)
                QFILE[v][r["_id"]] = strip_user(r["text"])

# turns: history (questions and agent answers before the current one), reference, human answerability
TASKS, bad_input = {}, 0
for ln in open(f"{REPO}/{H}/generation_tasks/reference.jsonl", encoding="utf-8"):
    if not ln.strip():
        continue
    r = json.loads(ln)
    inp = r.get("input") or []
    if not inp or inp[-1]["speaker"] != "user":      # a task must end with a user question
        bad_input += 1
        continue
    d = dom_of(r.get("Collection") or r.get("collection"))
    users = [m["text"].strip() for m in inp if m["speaker"] == "user"]
    tid = r["task_id"]
    t = {"task_id": tid, "conversation_id": r["conversation_id"], "turn": int(r["turn"]), "domain": d,
         "answerability": (r.get("Answerability") or r.get("answerability") or ["?"])[0],
         "query": users[-1], "history": qa_pairs(inp[:-1]),
         "reference": ((r.get("targets") or [{}])[0].get("text") or "").strip(),
         "gold": list(QRELS[d].get(tid, {})), "ts": (inp[-1].get("metadata") or {}).get("created_at"),
         "q_lastturn": QFILE["lastturn"].get(tid) or users[-1],
         "q_questions": QFILE["questions"].get(tid) or "\n".join(users),
         "rewrite_from_mtrag": tid in QFILE["rewrite"]}
    # only turns without qrels (UNANSWERABLE) lack MTRAG rewrites: for them the "rewrite" = all questions
    t["q_rewrite"] = QFILE["rewrite"].get(tid) or t["q_questions"]
    TASKS[tid] = t
LABEL = sorted(x for x, t in TASKS.items() if t["answerability"] in ("ANSWERABLE", "PARTIAL") and t["gold"])
UNANS = sorted(x for x, t in TASKS.items() if t["answerability"] == "UNANSWERABLE")
RET_TURNS = LABEL + UNANS
from collections import Counter
print("turns:", len(TASKS), dict(Counter(t["answerability"] for t in TASKS.values())))
print("for labels (ANSWERABLE + PARTIAL with qrels):", len(LABEL), dict(Counter(TASKS[x]["domain"] for x in LABEL)),
      "· unanswerable:", len(UNANS))
orphan = [q for d in MT_DOMAINS for q in QRELS[d] if q not in TASKS]
print("qrels without a task:", len(orphan), "· tasks not ending in a question (skipped):", bad_input,
      "· label turns without an MTRAG rewrite:",
      sum(not TASKS[x]["rewrite_from_mtrag"] for x in LABEL),
      "· last turn in file ≠ in task:", sum(TASKS[x]["q_lastturn"] != TASKS[x]["query"] for x in LABEL))
assert LABEL and len(orphan) <= 0.02 * sum(map(len, QRELS.values())), "turns and qrels do not match: check the formats"

# annotator judgments (conversations.json). A turn is matched by (timestamp, question text);
# the final judgment follows MTRAG's scripts/conversations2retrieval.py: the editor's if they judged, otherwise the author's
CONVS = json.load(open(f"{REPO}/{H}/conversations/conversations.json", encoding="utf-8"))
BY_TS = {(t["ts"], t["query"]): x for x, t in TASKS.items()}
HUMAN, n_user, unmatched, misturn = {}, 0, 0, 0
for c in CONVS:
    cur, k = None, 0
    for m in c.get("messages", []):
        if m.get("speaker") == "user":
            k += 1; n_user += 1
            cur = BY_TS.get((m.get("timestamp"), (m.get("text") or "").strip()))
            unmatched += cur is None
            misturn += cur is not None and TASKS[cur]["turn"] != k
        elif cur is not None:
            for cx in m.get("contexts") or []:
                fb = (cx.get("feedback") or {}).get("relevant") or {}
                who = c.get("editor") if c.get("editor") in fb else c.get("author")
                HUMAN.setdefault(cur, {})[cx["document_id"]] = {
                    "final": (fb.get(who) or {}).get("value"),
                    "votes": sorted(str((v or {}).get("value")) for v in fb.values()),
                    "text": cx.get("text") or ""}
print(f"conversations {len(CONVS)}, questions {n_user}: no matching task {unmatched}, turn number mismatch {misturn}")
_ans = [x for x in LABEL if TASKS[x]["answerability"] == "ANSWERABLE"]
_g = [(x, p) for x in _ans for p in TASKS[x]["gold"]]
print("ANSWERABLE gold with annotator judgment 'yes':", sum(HUMAN.get(x, {}).get(p, {}).get("final") == "yes"
                                                        for x, p in _g), "of", len(_g),
      "· 'no' passages in ANSWERABLE turns:", sum(h["final"] == "no" and p not in TASKS[x]["gold"]
                                          for x in _ans for p, h in HUMAN.get(x, {}).items()))

# corpora: read directly from the zip files, once per session
CORPUS = {}
def corpus(d):
    if d not in CORPUS:
        ids, texts = W.load_corpus(REPO, d)
        CORPUS[d] = (ids, texts, {p: i for i, p in enumerate(ids)})
    return CORPUS[d]

def ptext(d, pid):
    i = corpus(d)[2].get(pid)
    return None if i is None else CORPUS[d][1][i]

t0 = time.time()
for d in MT_DOMAINS:
    ids, _, pos = corpus(d)
    g = {p for x in LABEL if TASKS[x]["domain"] == d for p in TASKS[x]["gold"]}
    print(f"  {d:<7} passages {len(ids):>7} · gold in corpus {sum(p in pos for p in g)}/{len(g)}")
    assert sum(p in pos for p in g) >= 0.95 * len(g), f"{d}: corpus ids do not match qrels"
print(f"corpora in {time.time() - t0:.0f} s")

In [ ]:
# ── 5. BM25 (bm25s): last turn, all questions, rewritten question → Drive ──
import bm25s
try:
    import Stemmer
    STEM, STEM_NAME = Stemmer.Stemmer("english"), "PyStemmer english"
except ImportError:
    STEM, STEM_NAME = None, None
    print("WARNING: PyStemmer is not installed; BM25 runs without stemming (recorded in config.json)")
QSETS = {"bm25_lastturn": "q_lastturn", "bm25_questions": "q_questions", "bm25_rewrite": "q_rewrite"}
tokz = lambda xs: bm25s.tokenize(xs, stopwords="en", stemmer=STEM, show_progress=False)
for d in MT_DOMAINS:
    todo = [n for n in QSETS if not os.path.exists(run_path(n, d))]
    if not todo:
        print(f"[exists] BM25 {d}"); continue
    ids, texts, _ = corpus(d)
    t0 = time.time()
    bm = bm25s.BM25(method="lucene", k1=BM25_K1, b=BM25_B)
    bm.index(tokz(texts), show_progress=False)
    tids = [x for x in RET_TURNS if TASKS[x]["domain"] == d]
    for name in todo:
        if not tids:
            write_run(run_path(name, d), {}, name); continue
        docs, scores = bm.retrieve(tokz([TASKS[x][QSETS[name]] for x in tids]), k=min(TOPK, len(ids)),
                                   show_progress=False)
        # a score of 0 means no shared terms: such "hits" are not written
        write_run(run_path(name, d), {x: {ids[j]: float(s) for j, s in zip(docs[i], scores[i]) if s > 0}
                                      for i, x in enumerate(tids)}, name)
    del bm
    print(f"BM25 {d}: {len(ids)} passages, {len(tids)} turns × {len(todo)} queries in {(time.time() - t0) / 60:.1f} min")
for name, field in QSETS.items():
    json.dump({"retriever": "bm25s " + bm25s.__version__, "method": "lucene", "k1": BM25_K1, "b": BM25_B,
               "stopwords": "en", "stemmer": STEM_NAME, "query": field, "topk": TOPK, "mtrag_rev": MTRAG_REV,
               "note": "query text without '|user|:' prefixes; q_rewrite of turns without an MTRAG rewrite "
                       "(UNANSWERABLE) = q_questions", "date": time.strftime("%Y-%m-%dT%H:%M:%S")},
              open(f"{OUT}/runs/{name}/config.json", "w"), indent=2)

In [ ]:
# ── 6. Dense retrieval (Qwen3-Embedding-0.6B) on the rewritten question, and hybrid ──
# Vectors are computed by a separate process (this frees the GPU for the judge); the search runs here, numpy fp32.
SLUG = re.sub(r"[^A-Za-z0-9._-]", "_", EMBED_MODEL)
IDX = f"{OUT}/index/{SLUG}"
os.makedirs(IDX, exist_ok=True)
QUERIES = {d: [[x, TASKS[x]["q_rewrite"]] for x in RET_TURNS if TASKS[x]["domain"] == d] for d in MT_DOMAINS}
QPATH = {d: f"{IDX}/q_rewrite_{d}_" + hashlib.sha1(json.dumps([EMBED_MODEL, MAX_Q_LEN, QUERIES[d]]).encode()).hexdigest()[:12]
            + ".npy" for d in MT_DOMAINS}
json.dump({"repo": REPO, "domains": MT_DOMAINS, "index": IDX, "model": EMBED_MODEL, "batch": EMB_BATCH,
           "max_doc_len": MAX_DOC_LEN, "max_q_len": MAX_Q_LEN, "dtype": DTYPE, "queries": QUERIES, "q_path": QPATH},
          open("/content/mtrag_embed.json", "w"))
# vectors are needed only for dense runs not computed yet (the vectors may be deleted to save Drive space)
NEED_VEC = not all(os.path.exists(run_path(n, d)) for n in ("dense_rewrite", "hybrid_rewrite") for d in MT_DOMAINS)
if NEED_VEC and not all(os.path.exists(p) for p in [f"{IDX}/{d}.npy" for d in MT_DOMAINS] + list(QPATH.values())):
    assert stream([sys.executable, "/content/mtrag_worker.py", "embed", "/content/mtrag_embed.json"]) == 0, \
        "embedder failed, see the log above"
EMB_CFG = (json.load(open(f"{IDX}/config.json")) if os.path.exists(f"{IDX}/config.json")
           else {"model": EMBED_MODEL, "note": "vectors deleted, runs are ready"})

def minmax_fuse(parts):
    """parts: [(weight, {doc: score})] → {doc: Σ weight·minmax(score)}; absent from a list = 0 (as in fuse.py and analysis)."""
    s = {}
    for w, r in parts:
        if r:
            lo, hi = min(r.values()), max(r.values())
            for d, v in r.items():
                s[d] = s.get(d, 0.0) + w * ((v - lo) / (hi - lo) if hi > lo else 1.0)
    return dict(sorted(s.items(), key=lambda x: -x[1])[:TOPK])

for d in MT_DOMAINS:
    if not os.path.exists(run_path("dense_rewrite", d)):
        ids = json.load(open(f"{IDX}/{d}.ids.json"))
        E, Q = np.load(f"{IDX}/{d}.npy").astype(np.float32), np.load(QPATH[d]).astype(np.float32)
        run = {}
        for b in range(0, len(Q), 256):
            sc = Q[b:b + 256] @ E.T                   # vectors are normalized: dot product = cosine
            k = min(TOPK, sc.shape[1])
            top = np.argpartition(-sc, k - 1, axis=1)[:, :k]
            for i, (x, _) in enumerate(QUERIES[d][b:b + 256]):
                j = top[i][np.argsort(-sc[i, top[i]], kind="stable")]
                run[x] = {ids[c]: float(sc[i, c]) for c in j}
        write_run(run_path("dense_rewrite", d), run, "dense_rewrite")
        del E
    if not os.path.exists(run_path("hybrid_rewrite", d)):
        a, b = read_run(run_path("bm25_rewrite", d)), read_run(run_path("dense_rewrite", d))
        write_run(run_path("hybrid_rewrite", d),
                  {x: minmax_fuse([(HYB_W, a.get(x, {})), (1 - HYB_W, b.get(x, {}))]) for x in sorted(set(a) | set(b))},
                  "hybrid_rewrite")
json.dump({**EMB_CFG, "query": "q_rewrite", "topk": TOPK, "search": "cosine, numpy fp32", "mtrag_rev": MTRAG_REV},
          open(f"{OUT}/runs/dense_rewrite/config.json", "w"), ensure_ascii=False, indent=2)
json.dump({"fusion": "minmax", "parts": {"bm25_rewrite": HYB_W, "dense_rewrite": 1 - HYB_W}, "topk": TOPK,
           "missing": 0, "weight": "fixed a priori: MTRAG has no train split", "mtrag_rev": MTRAG_REV},
          open(f"{OUT}/runs/hybrid_rewrite/config.json", "w"), indent=2)
print("dense and hybrid ready:", f"{OUT}/runs")

In [ ]:
# ── 7. Retrieval ladder: nDCG@10, R@5, R@100, macro over domains and micro over turns ──
RETRIEVERS = ["bm25_lastturn", "bm25_questions", "bm25_rewrite", "dense_rewrite", "hybrid_rewrite"]
DOMS_L = [d for d in MT_DOMAINS if any(TASKS[x]["domain"] == d for x in LABEL)]
PER_TURN, RET_ROWS = {}, []
for name in RETRIEVERS:
    for d in DOMS_L:
        qr = {x: QRELS[d][x] for x in LABEL if TASKS[x]["domain"] == d}
        run = read_run(run_path(name, d))
        ev = pytrec_eval.RelevanceEvaluator(qr, {"ndcg_cut.10", "recall.5", "recall.100"}).evaluate(
            {x: run[x] for x in qr if run.get(x)})
        for x in qr:              # a turn without a run scores 0, as on the leaderboard
            e = ev.get(x, {})
            PER_TURN[(name, x)] = {"ndcg10": e.get("ndcg_cut_10", 0.0), "r5": e.get("recall_5", 0.0),
                                   "r100": e.get("recall_100", 0.0)}
    row = {"retriever": name, "n": len(LABEL)}
    for m in ("ndcg10", "r5", "r100"):
        dm = {d: float(np.mean([PER_TURN[(name, x)][m] for x in LABEL if TASKS[x]["domain"] == d])) for d in DOMS_L}
        row[f"{m}_macro"] = round(float(np.mean(list(dm.values()))), 4)
        row[f"{m}_micro"] = round(float(np.mean([PER_TURN[(name, x)][m] for x in LABEL])), 4)
        row.update({f"{m}_{d}": round(v, 4) for d, v in dm.items()})
    RET_ROWS.append(row)
with open(f"{OUT}/retrieval.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(RET_ROWS[0])); w.writeheader(); w.writerows(RET_ROWS)
print(f"{'retriever':<16}{'nDCG@10 macro':>14}{'micro':>8}{'R@5 macro':>11}{'micro':>8}{'R@100':>8}  "
      + "".join(f"{d:>8}" for d in DOMS_L))
for r in RET_ROWS:
    print(f"{r['retriever']:<16}{r['ndcg10_macro']:>14.4f}{r['ndcg10_micro']:>8.4f}{r['r5_macro']:>11.4f}"
          f"{r['r5_micro']:>8.4f}{r['r100_macro']:>8.4f}  " + "".join(f"{r[f'ndcg10_{d}']:>8.4f}" for d in DOMS_L))
print("MTRAG README (macro nDCG@10, last turn / rewrite): BM25 .21/.25, BGE-base .30/.38, ELSER .49/.54")
print("ladder by nDCG@10:", " < ".join(r["retriever"] for r in sorted(RET_ROWS, key=lambda r: r["ndcg10_macro"])),
      f"→ {OUT}/retrieval.csv")

In [ ]:
# ── 8. Contexts: top 5 of each retriever (+ all gold); files for local analysis ──
CTX = {name: {} for name in RETRIEVERS}
for name in RETRIEVERS:
    for d in MT_DOMAINS:
        top = G.read_top(run_path(name, d), K_CTX)
        for x in RET_TURNS:
            if TASKS[x]["domain"] == d:
                CTX[name][x] = top.get(x, [])
if DO_GOLD:     # gold missing from the corpus does not enter the context (no text)
    CTX["gold"] = {x: [p for p in TASKS[x]["gold"] if ptext(TASKS[x]["domain"], p) is not None] for x in LABEL}
    print("gold passages without text in the corpus:", sum(len(TASKS[x]["gold"]) - len(CTX["gold"][x]) for x in LABEL))
JOBS = RETRIEVERS + (["gold"] if DO_GOLD else [])
QLAB = {job: {x: G.sufficiency(TASKS[x]["gold"], CTX[job][x]) for x in LABEL} for job in JOBS}   # (n in context, all/part/none)
for job in JOBS:
    c = Counter(v[1] for v in QLAB[job].values())
    print(f"  {job:<16} qrels label, top-{K_CTX}: all {c['all']:>4}  part {c['part']:>4}  none {c['none']:>4}")

save_jsonl(f"{OUT}/turns.jsonl", [
    {**{k: v for k, v in TASKS[x].items() if k != "ts"}, "use": "label" if x in LABEL else "unans",
     "human": {p: h["final"] for p, h in HUMAN.get(x, {}).items()},
     "top5": {name: CTX[name].get(x, []) for name in RETRIEVERS}} for x in RET_TURNS])
need = {(TASKS[x]["domain"], p) for job in CTX for x, ids in CTX[job].items() for p in ids}
need |= {(TASKS[x]["domain"], p) for x in HUMAN for p in HUMAN[x]}
save_jsonl(f"{OUT}/passages.jsonl", [
    {"domain": d, "doc_id": p, "in_corpus": ptext(d, p) is not None, "page": page_of(d, p),
     "text": ptext(d, p) if ptext(d, p) is not None else W.norm(next((HUMAN[x][p]["text"] for x in HUMAN if p in HUMAN[x]), ""))}
    for d, p in sorted(need)])
save_jsonl(f"{OUT}/unanswerable_hybrid_top5.jsonl", [
    {"task_id": x, "domain": TASKS[x]["domain"], "turn": TASKS[x]["turn"], "query": TASKS[x]["query"],
     "history": TASKS[x]["history"], "reference": TASKS[x]["reference"], "query_used": TASKS[x]["q_rewrite"],
     "rewrite_from_mtrag": TASKS[x]["rewrite_from_mtrag"],
     "top5": [{"doc_id": p, "text": ptext(TASKS[x]["domain"], p), "human": HUMAN.get(x, {}).get(p, {}).get("final")}
              for p in CTX["hybrid_rewrite"].get(x, [])]} for x in UNANS])
print(f"turns → turns.jsonl, {len(need)} passages → passages.jsonl, {len(UNANS)} unanswerable → unanswerable_hybrid_top5.jsonl")

In [ ]:
# ── 9. phi-4 judge: each context and (DO_SINGLE) each passage on its own → Drive/reteco/mtrag/suff ──
# Prompt and parsing exactly as in suff.ipynb (suff_prompt.py); reference = the MTRAG answer (targets); documents [1]…[5] without ids.
JSHORT = JUDGE.split("/")[-1]

def tview(x, ref):
    t = TASKS[x]
    return {"history": t["history"], "query": t["query"], **({"reference": t["reference"]} if ref else {})}

def meta(x, job, ids, lab_key):
    t = TASKS[x]
    n, lab = G.sufficiency(t["gold"], ids) if t["gold"] else (0, "unans")
    return {"task_id": x, "domain": t["domain"], "turn": t["turn"], "answerability": t["answerability"],
            "job": job, "context_ids": ids, "n_gold": len(t["gold"]), "gold_in_ctx": n, lab_key: lab}

GOLD_OF_CONV = {}
for x in LABEL:
    GOLD_OF_CONV.setdefault(TASKS[x]["conversation_id"], {}).update({p: x for p in TASKS[x]["gold"]})
n_items = Counter()
with open("/content/mtrag_judge_items.jsonl", "w", encoding="utf-8") as f:
    for job in JOBS:
        for x in LABEL:
            ids = CTX[job][x]
            f.write(json.dumps({"job": job, "key": x, "t": tview(x, True), "meta": meta(x, job, ids, "suff_qrels"),
                                "docs": [ptext(TASKS[x]["domain"], p) for p in ids]}, ensure_ascii=False) + "\n")
            n_items[job] += 1
    for x in (LABEL if DO_SINGLE else []):
        t = TASKS[x]
        if t["answerability"] not in SINGLE_CLASSES:
            continue
        gold, pages = set(t["gold"]), {page_of(t["domain"], p) for p in t["gold"]}
        conv_gold = GOLD_OF_CONV.get(t["conversation_id"], {})
        cands = [(p, "gold") for p in t["gold"]] + [(p, "human_no") for p, h in HUMAN.get(x, {}).items()
                                                    if h["final"] == "no" and p not in gold]
        for p, kind in cands:
            txt = ptext(t["domain"], p)
            txt = txt if txt is not None else W.norm(HUMAN.get(x, {}).get(p, {}).get("text"))
            if not txt:
                continue
            h = HUMAN.get(x, {}).get(p, {})
            m = {**meta(x, "single", [p], "suff_qrels"), "doc_id": p, "kind": kind, "human_final": h.get("final"),
                 "human_votes": h.get("votes"), "in_corpus": ptext(t["domain"], p) is not None,
                 "same_page_as_gold": kind == "human_no" and page_of(t["domain"], p) in pages,
                 "gold_other_turn": kind == "human_no" and conv_gold.get(p, x) != x}
            f.write(json.dumps({"job": "single", "key": f"{x}|{p}", "t": tview(x, True), "meta": m, "docs": [txt]},
                               ensure_ascii=False) + "\n")
            n_items[f"single·{kind}"] += 1
print("judge items:", dict(n_items))
assert BIG, "phi-4 in bf16 needs ≈29 GB: Runtime → Change runtime type → A100"
JCFG = {"judge": JUDGE, "judge_items": "/content/mtrag_judge_items.jsonl", "out": OUT, "dtype": DTYPE,
        "maxlen": MAXLEN, "judge_new": JUDGE_NEW, "trim_words": TRIM_WORDS, "chunk": CHUNK}
json.dump(JCFG, open("/content/mtrag_judge.json", "w"))
# if the judge fails, the readers still run (they do not need labels); the summary reports what is missing
free_gpu()
JUDGE_FAILED = stream([sys.executable, "/content/mtrag_worker.py", "judge", "/content/mtrag_judge.json"]) != 0
if JUDGE_FAILED:
    print("!!! judge failed, see the log above; readers and summary continue, labels will be incomplete")
json.dump({**JCFG, "failed": JUDGE_FAILED, "jobs": JOBS, "k_ctx": K_CTX, "single_classes": SINGLE_CLASSES,
           "system": S.SYSTEM, "task": S.TASK, "reference": "MTRAG targets[0].text", "gpu": torch.cuda.get_device_name(0),
           "date": time.strftime("%Y-%m-%dT%H:%M:%S")}, open(f"{OUT}/suff/config.json", "w"), ensure_ascii=False, indent=2)

In [ ]:
# ── 10. Summary (no GPU needed): labels by retriever, human check of the judge, readers ──
# Bootstrap 1000 × seed 0, 95% percentile: labels and readers are resampled over turns within a domain,
# paired (the same turns in all contexts); the human check resamples turns as clusters.
# The judge's partial is excluded from P/R and AUROC (as in the paper); its share is reported separately.
from scipy.stats import rankdata
NAN = float("nan")
KEYS = list(LABEL)
DOMS = np.array([TASKS[x]["domain"] for x in KEYS])
STRATA = [np.where(DOMS == d)[0] for d in MT_DOMAINS if (DOMS == d).any()]

def keyed(path):
    return {r["key"]: r for r in W.jsonl(path)}

def ratio(a, b):
    return float(a) / float(b) if b else NAN

def auroc(y, s):
    """AUROC via ranks (ties get the average rank, as in roc_auc_score); nan if one of the classes is missing."""
    y, s = np.asarray(y, bool), np.asarray(s, float)
    n1, n0 = int(y.sum()), int((~y).sum())
    if not n1 or not n0:
        return NAN
    return float((rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def boot(fn, n_items, strata=None, clusters=None):
    """fn(idx) → {metric: value}; returns {metric: (point, lo, hi)}."""
    point = fn(np.arange(n_items))
    rng, acc = np.random.default_rng(SEED), {k: [] for k in point}
    for _ in range(N_BOOT):
        if strata is not None:
            idx = np.concatenate([g[rng.integers(0, len(g), len(g))] for g in strata])
        else:
            idx = np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])
        for k, v in fn(idx).items():
            acc[k].append(v)
    out = {}
    for k, v in point.items():
        a = np.asarray(acc[k], float); a = a[~np.isnan(a)]
        out[k] = (v, *((float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))) if len(a) else (NAN, NAN)))
    return out

FW = 21           # width of a "value [CI]" column
def fmt(t, w=FW):
    v, lo, hi = t
    return f"{'—':>{w}}" if np.isnan(v) else f"{f'{v:.3f} [{lo:.2f},{hi:.2f}]':>{w}}"

def label_metrics(job, idx):
    q, lab = QARR[job][idx], LAB[job][idx]
    ok = lab != ""                                   # the judge label was parsed
    q, lab, sp = q[ok], lab[ok], EVSP[job][idx][ok]
    none, yes, par, no = q == "none", lab == "yes", lab == "partial", lab == "no"
    tp, fp, fn = (none & no).sum(), (none & yes).sum(), (~none & no).sum()
    m = {"n": float(ok.sum()), "share_qrels_none": float(none.mean()) if len(q) else NAN,
         "share_judge_no": float(no.mean()) if len(q) else NAN,
         "share_judge_partial": float(par.mean()) if len(q) else NAN,
         "share_judge_yes": float(yes.mean()) if len(q) else NAN,
         "false_miss": ratio((none & yes).sum(), none.sum()),
         "false_miss_yes_or_partial": ratio((none & (yes | par)).sum(), none.sum()),
         "P_none_detects_no": ratio(tp, tp + fp), "R_none_detects_no": ratio(tp, tp + fn),
         "overestimate": ratio(none.mean(), no.mean()) if len(q) else NAN,
         "false_miss_same_page": ratio((none & yes & sp).sum(), (none & yes).sum())}
    if job in RETRIEVERS:
        nd, dm = NDARR[job][idx], DOMS[idx]
        m["ndcg10"] = float(np.mean([nd[dm == d].mean() for d in np.unique(dm)]))
        m["gold_in_top5"] = float((QARR[job][idx] != "none").mean())
    return m

def summarize(final):
    global QARR, LAB, EVSP, NDARR
    rows = []
    def add(section, reader, res):
        for k, (v, lo, hi) in res.items():
            job, metric = k.rsplit("|", 1)
            rows.append({"section": section, "job": job, "reader": reader, "metric": metric,
                         "value": None if np.isnan(v) else round(v, 4),
                         "lo": None if np.isnan(lo) else round(lo, 4), "hi": None if np.isnan(hi) else round(hi, 4)})
    QARR = {j: np.array([QLAB[j][x][1] for x in KEYS]) for j in JOBS}
    NDARR = {j: np.array([PER_TURN[(j, x)]["ndcg10"] for x in KEYS]) for j in RETRIEVERS}
    LAB, EVSP = {}, {}
    for job in JOBS:
        recs = keyed(f"{OUT}/suff/{JSHORT}_{job}.jsonl")
        LAB[job] = np.array([(recs.get(x) or {}).get("label") or "" for x in KEYS])
        sp = []
        for x in KEYS:
            r, t = recs.get(x) or {}, TASKS[x]
            ids, gold = r.get("context_ids") or [], set(t["gold"])
            pages = {page_of(t["domain"], p) for p in gold}
            sp.append(any(1 <= e <= len(ids) and ids[e - 1] not in gold and page_of(t["domain"], ids[e - 1]) in pages
                          for e in r.get("evidence_docs") or []))
        EVSP[job] = np.array(sp, bool)
    have = [j for j in JOBS if (LAB[j] != "").any()]

    # (a) labels: qrels none vs. the judge, per retriever
    if have:
        rets = [j for j in RETRIEVERS if j in have]
        nd0 = {j: label_metrics(j, np.arange(len(KEYS)))["ndcg10"] for j in rets}
        weak = min(rets, key=nd0.get) if rets else None
        def fn(idx):
            out = {f"{j}|{k}": v for j in have for k, v in label_metrics(j, idx).items()}
            for j in rets:
                if j != weak:
                    for k in ("ndcg10", "share_qrels_none", "share_judge_no", "false_miss", "P_none_detects_no"):
                        out[f"{j} − {weak}|{k}"] = out[f"{j}|{k}"] - out[f"{weak}|{k}"]
            return out
        res = boot(fn, len(KEYS), strata=STRATA)
        add("labels", "", res)
        print(f"(a) qrels label 'none' (no gold in top-{K_CTX}) vs. the {JSHORT} judge; partial excluded from P/R")
        print(f"{'context':<16}{'n':>5}{'nDCG@10':>{FW}}{'qrels none':>{FW}}{'judge no':>{FW}}{'partial':>8}{'yes':>7}"
              f"{'false miss':>{FW}}{'P(none→no)':>{FW}}{'R(none→no)':>{FW}}{'same page':>11}")
        for j in sorted(have, key=lambda j: nd0.get(j, 9)):
            g = lambda k: res.get(f"{j}|{k}", (NAN, NAN, NAN))
            print(f"{j:<16}{g('n')[0]:>5.0f}{fmt(g('ndcg10'))}{fmt(g('share_qrels_none'))}{fmt(g('share_judge_no'))}"
                  f"{g('share_judge_partial')[0]:>8.3f}{g('share_judge_yes')[0]:>7.3f}{fmt(g('false_miss'))}"
                  f"{fmt(g('P_none_detects_no'))}{fmt(g('R_none_detects_no'))}{g('false_miss_same_page')[0]:>11.3f}")
        print("false miss = share of judge-yes among qrels-none; 'same page' = among those, evidence from the same page"
              "\nas the gold (a neighbouring chunk). Differences from the weakest retriever (" + str(weak) + "), paired bootstrap:")
        for j in rets:
            if j != weak:
                g = lambda k: res[f"{j} − {weak}|{k}"]
                print(f"  {j:<16} ΔnDCG {fmt(g('ndcg10'))}  ΔP {fmt(g('P_none_detects_no'))}  "
                      f"Δfalse miss {fmt(g('false_miss'))}  Δqrels none {fmt(g('share_qrels_none'))}")
    else:
        print("(a) no judge labels yet: run cell 9 first")

    # (b) human check of the judge: single passages
    sg = [r for r in W.jsonl(f"{OUT}/suff/{JSHORT}_single.jsonl") if r.get("label")]
    if sg:
        kind, lab = np.array([r["kind"] for r in sg]), np.array([r["label"] for r in sg])
        spg = np.array([bool(r.get("same_page_as_gold")) for r in sg])
        oth = np.array([bool(r.get("gold_other_turn")) for r in sg])
        # group names are the "job" values of section human_single in summary.csv: gold passages; passages the
        # annotators marked not relevant (human_no); human_no on the same page as a gold passage or on another
        # page; human_no that is the gold of another turn
        groups = {"gold (annotator: relevant)": kind == "gold", "human_no (annotator: not relevant)": kind == "human_no",
                  "human_no · same page": (kind == "human_no") & spg,
                  "human_no · other page": (kind == "human_no") & ~spg,
                  "human_no · gold of another turn": (kind == "human_no") & oth}
        tid = np.array([r["task_id"] for r in sg])
        clusters = [np.where(tid == t)[0] for t in np.unique(tid)]
        def fn(idx):
            out = {}
            for g, m in groups.items():
                l = lab[idx][m[idx]]
                out[f"{g}|n"] = float(len(l))
                out[f"{g}|judge_yes"] = float((l == "yes").mean()) if len(l) else NAN
                out[f"{g}|judge_yes_or_partial"] = float(np.isin(l, ["yes", "partial"]).mean()) if len(l) else NAN
            return out
        res = boot(fn, len(sg), clusters=clusters)
        add("human_single", "", res)
        print(f"\n(b) judge on single passages, turns {SINGLE_CLASSES}: P(judge yes | MTRAG annotator judgment)")
        print(f"{'group':<36}{'n':>6}{'judge yes':>{FW}}{'yes or partial':>{FW}}")
        for g in groups:
            print(f"{g:<36}{res[f'{g}|n'][0]:>6.0f}{fmt(res[f'{g}|judge_yes'])}{fmt(res[f'{g}|judge_yes_or_partial'])}")
        print("a passage rejected by the annotator but judged 'yes' is either a judge error or a gap in the human judgments;\n"
              "a single gold passage can be partial when the answer is assembled from several passages.")
    elif DO_SINGLE:
        print("\n(b) missing:", f"{OUT}/suff/{JSHORT}_single.jsonl")

    # (c) readers: refusals and AUROC of p_answerable under the qrels label and under the judge label
    for reader in READERS:
        short = reader.split("/")[-1]
        gen = {j: keyed(f"{OUT}/gen/{short}_{j}.jsonl") for j in JOBS}
        gen = {j: g for j, g in gen.items() if g}
        if not gen:
            if final:
                print(f"\n(c) {short}: no answers")
            continue
        A = {j: (np.array([float(g[x]["status"].startswith("UN")) if x in g else NAN for x in KEYS]),
                 np.array([g[x]["p_answerable"] if x in g and g[x]["p_answerable"] is not None else NAN for x in KEYS],
                          float)) for j, g in gen.items()}
        def fn(idx):
            out = {}
            for j, (ref, p) in A.items():
                r, pp, q, lb = ref[idx], p[idx], QARR[j][idx], LAB[j][idx]
                h = ~np.isnan(r)
                rate = lambda m: float(r[h & m].mean()) if (h & m).any() else NAN
                out[f"{j}|n"] = float(h.sum())
                out[f"{j}|refusal"] = rate(np.ones(len(idx), bool))
                out[f"{j}|refusal_S_q"], out[f"{j}|refusal_I_q"] = rate(q == "all"), rate(q == "none")
                out[f"{j}|refusal_S_c"], out[f"{j}|refusal_I_c"] = rate(lb == "yes"), rate(lb == "no")
                ok = h & ~np.isnan(pp)
                mq, mc = ok & np.isin(q, ["all", "none"]), ok & np.isin(lb, ["yes", "no"])
                out[f"{j}|auroc_qrels"] = auroc(q[mq] == "all", pp[mq])
                out[f"{j}|auroc_content"] = auroc(lb[mc] == "yes", pp[mc])
                out[f"{j}|d_auroc"] = out[f"{j}|auroc_content"] - out[f"{j}|auroc_qrels"]
                if "gold" in A and j != "gold":      # 2b/2c contrast: refusal on insufficient context minus on gold, same turns
                    rg = A["gold"][0][idx]
                    for nm, m in (("contrast_q", q == "none"), ("contrast_c", lb == "no")):
                        mm = h & ~np.isnan(rg) & m
                        out[f"{j}|{nm}"] = float(r[mm].mean() - rg[mm].mean()) if mm.any() else NAN
            return out
        res = boot(fn, len(KEYS), strata=STRATA)
        add("readers", short, res)
        print(f"\n(c) {short}: refusal = status UNANSWERABLE; S/I = sufficient/insufficient context by qrels (q: all/none)"
              f"\n    and by the judge (c: yes/no); AUROC of p_answerable; contrast = refusal on I minus on gold, same turns")
        print(f"{'context':<16}{'n':>5}{'refusal':>7}{'S_q':>7}{'I_q':>7}{'S_c':>7}{'I_c':>7}"
              f"{'AUROC qrels':>{FW}}{'AUROC judge':>{FW}}{'Δ':>{FW}}{'contrast q':>{FW}}{'contrast c':>{FW}}")
        for j in [j for j in JOBS if j in A]:
            g = lambda k: res.get(f"{j}|{k}", (NAN, NAN, NAN))
            print(f"{j:<16}{g('n')[0]:>5.0f}" + "".join(f"{g(k)[0]:>7.2f}" for k in
                  ("refusal", "refusal_S_q", "refusal_I_q", "refusal_S_c", "refusal_I_c"))
                  + "".join(fmt(g(k)) for k in ("auroc_qrels", "auroc_content", "d_auroc", "contrast_q", "contrast_c")))
        gu = keyed(f"{OUT}/gen/{short}_unans_hybrid.jsonl")
        us = [x for x in UNANS if x in gu]
        if us:
            ru = np.array([float(gu[x]["status"].startswith("UN")) for x in us])
            du = np.array([TASKS[x]["domain"] for x in us])
            ru_res = boot(lambda idx: {"unans_hybrid|refusal": float(ru[idx].mean())}, len(us),
                          strata=[np.where(du == d)[0] for d in MT_DOMAINS if (du == d).any()])
            add("readers", short, ru_res)
            print(f"{'unans_hybrid':<16}{len(us):>5}{ru_res['unans_hybrid|refusal'][0]:>7.2f}  ← UNANSWERABLE turns "
                  f"(human label), hybrid top 5; CI {fmt(ru_res['unans_hybrid|refusal']).strip()}")

    if rows:
        with open(f"{OUT}/summary.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
        print(f"\n{len(rows)} numbers → {OUT}/summary.csv")
    return rows

SUMMARY = summarize(final=False)     # the labels are ready even without readers

In [ ]:
# ── 11. Readers: each in a separate process (classes, prompts and p_answerable from gen_worker) ──
# Contexts: top 5 of each retriever, all gold (DO_GOLD) and the hybrid top 5 for UNANSWERABLE turns.
n_items = Counter()
with open("/content/mtrag_read_items.jsonl", "w", encoding="utf-8") as f:
    for job in JOBS:
        for x in LABEL:
            ids = CTX[job][x]
            f.write(json.dumps({"job": job, "key": x, "t": tview(x, False), "meta": meta(x, job, ids, "suff"),
                                "docs": [ptext(TASKS[x]["domain"], p) for p in ids]}, ensure_ascii=False) + "\n")
            n_items[job] += 1
    for x in UNANS:
        ids = CTX["hybrid_rewrite"].get(x, [])
        f.write(json.dumps({"job": "unans_hybrid", "key": x, "t": tview(x, False),
                            "meta": meta(x, "unans_hybrid", ids, "suff"),
                            "docs": [ptext(TASKS[x]["domain"], p) for p in ids]}, ensure_ascii=False) + "\n")
        n_items["unans_hybrid"] += 1
RCFG = {"read_items": "/content/mtrag_read_items.jsonl", "out": OUT, "dtype": DTYPE, "maxlen": MAXLEN,
        "read_new": READ_NEW, "trim_words": TRIM_WORDS, "chunk": CHUNK}
json.dump(RCFG, open("/content/mtrag_read.json", "w"))
print("items per reader:", dict(n_items), "· readers:", READERS or "none (READERS = [])")
FAILED = []
for r in READERS:
    print(f"\n=== {r} ===")
    free_gpu()
    if stream([sys.executable, "/content/mtrag_worker.py", "read", r, "/content/mtrag_read.json"]) != 0:
        FAILED.append(r)
        print(f"!!! {r} failed, see the log above; the remaining readers and the summary continue")
if READERS:
    json.dump({**RCFG, "readers": READERS, "failed": FAILED, "jobs": JOBS + ["unans_hybrid"], "k_ctx": K_CTX,
               "gpu": torch.cuda.get_device_name(0), "date": time.strftime("%Y-%m-%dT%H:%M:%S")},
              open(f"{OUT}/gen/config.json", "w"), ensure_ascii=False, indent=2)

In [ ]:
# ── 12. Final summary (with readers) and the overall config.json ────────────
SUMMARY = summarize(final=True)
json.dump({"notebook": "mtrag.ipynb", "mtrag_rev": MTRAG_REV, "domains": MT_DOMAINS, "n_label_turns": len(LABEL),
           "n_unans_turns": len(UNANS), "retrievers": RETRIEVERS, "jobs": JOBS, "embed_model": EMBED_MODEL,
           "max_doc_len": MAX_DOC_LEN, "bm25": {"k1": BM25_K1, "b": BM25_B, "stemmer": STEM_NAME}, "hyb_w": HYB_W,
           "topk": TOPK, "k_ctx": K_CTX, "judge": JUDGE, "judge_failed": JUDGE_FAILED, "readers": READERS,
           "failed_readers": FAILED,
           "do_gold": DO_GOLD, "do_single": DO_SINGLE, "single_classes": SINGLE_CLASSES, "maxlen": MAXLEN,
           "judge_new": JUDGE_NEW, "read_new": READ_NEW, "trim_words": TRIM_WORDS, "n_boot": N_BOOT, "seed": SEED,
           "dtype": DTYPE, "gpu": torch.cuda.get_device_name(0), "date": time.strftime("%Y-%m-%dT%H:%M:%S")},
          open(f"{OUT}/config.json", "w"), ensure_ascii=False, indent=2)
if JUDGE_FAILED or FAILED:
    print("!!! INCOMPLETE:", "judge" if JUDGE_FAILED else "", *FAILED,
          "— 'Run all' again: finished parts are skipped")
print("Next: copy", OUT, "(without index/) from Drive to data/drive/reteco/mtrag/ of the repo.")

In [ ]:
# ── End. Everything is on Drive; the runtime disconnects so that no compute units are spent ──
print("done:", time.strftime("%Y-%m-%d %H:%M:%S"), "· log:", globals().get("LOG_PATH", "—"))
if AUTO_STOP:
    stop_runtime("notebook finished")